## 📋 **Scenario: Gestione Giacimento - Perforazione, Assay e Geotecnica**
### Un progetto di esplorazione ha raccolto dati da **sondaggi**, **analisi geochimiche** e **test geotecnici** per valutare la fattibilità di una miniera a cielo aperto.
    
## 🎯 **QUESITI (con JOIN e CTE)**

### **A. Pulizia Dati**
**1. Crea 3 tabelle temporanee** con:
   - ID uniformi (uppercase, formato standard)
   - Date in `YYYY-MM-DD`
   - Valori numerici (gestisci `zero`, `ND`, spazi)
   - Testi in uppercase e trim

### **B. Analisi con JOIN e CTE**

**1. Report completo sondaggi**  
Per ogni sondaggio, mostra:
- `ID_Sondaggio`, `Sito`, `Profondità_m`, `Metodo`
- `num_campioni` (da ASSAY)
- `num_test` (da GEOTECNICA)
- `media_Au_ppm`
- `media_RQD`

**2. Classifica siti per qualità roccia**  
Calcola per ogni sito:
- `media_RQD`
- `media_Au_ppm`
- `classifica` (usando CTE: RQD > 75 = "Buona", 50-75 = "Media", < 50 = "Scarsa")

**3. Anomalie da investigare**  
Trova sondaggi con:
- `RQD < 50` MA `Au_ppm > 1.0` (roccia scadente ma mineralizzata)

**4. Profondità ottimale**  
Usando CTE, calcola per ogni sito:
- `profondità_media_Au_max` (profondità dove Au è massimo)
- `RQD_corrispondente`

**5. Efficienza perforazione**  
Per ogni `Metodo`, calcola:
- `media_profondità`
- `media_Au_ppm`
- `media_RQD`
- `num_sondaggi`

---
    
## ✅ **Obiettivi**

1. **Pulisci** le 3 tabelle (crea tabelle temporanee)
2. **JOIN** per rispondere ai 5 quesiti
3. **CTE** almeno in uno dei quesiti (suggerito quesiti 2, 3, 4)
4. **Scrivi tu le considerazioni finali** basate sui risultati

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tabella 1: SONDAGGI
sondaggi = {
    'ID_Sondaggio': ['DH-101', 'dh-101', 'DH102 ', 'DH-103', 'DH-104'],
    'Data': ['2024-01-15', '20/02/2024', '2024.03.10', '2024-04-05', '15/05/2024'],
    'Sito': ['Nord', 'nord', 'SUD', 'sud', 'Est'],
    'Profondità_m': ['150', '200.5', 'zero', '175', '250'],
    'Metodo': ['RC', 'rc', 'Diamond', 'DIAMOND', 'RC']
}

# Tabella 2: ASSAY
assay = {
    'ID_Campione': ['S-101', 's-101', 'S102', 'S103 ', 'S104', 'S105'],
    'ID_Sondaggio': ['DH-101', 'DH101', 'DH-102', 'DH-102', 'DH-103', 'DH-104'],
    'Da_m': ['0.0', '10.5', '15', '20', '25', '30'],
    'A_m': ['1.0', '11.5', '16', '25', '30', '35'],
    'Au_ppm': ['0.3', '1.5', 'ND', '2.1', '0.8', '1.2'],
    'Cu_ppm': ['120', '250', 'ND', '85', '150', '200']
}

# Tabella 3: GEOTECNICA
geotecnica = {
    'ID_Test': ['GT-01', 'gt-01', 'GT02', 'GT-03', 'GT-04'],
    'ID_Sondaggio': ['DH-101', 'DH101', 'DH-102', 'DH-103', 'DH-104'],
    'Profondità_m': ['25', '50.5', '75', 'zero', '100'],
    'RQD': ['75', '85', 'ND', '92', '65'],
    'Tipo_Roccia': ['Granito', 'granito', 'SCISTO', 'Marmo', 'Granito']
}

# Carica in SQLite
df_sondaggi = pd.DataFrame(sondaggi)
df_assay = pd.DataFrame(assay)
df_geotecnica = pd.DataFrame(geotecnica)

df_sondaggi.to_sql('sondaggi', conn, index=False, if_exists='replace')
df_assay.to_sql('assay', conn, index=False, if_exists='replace')
df_geotecnica.to_sql('geotecnica', conn, index=False, if_exists='replace')

print("=== DATI CARICATI ===")
print("\nSONDAGGI:")
print(df_sondaggi)
print("\nASSAY:")
print(df_assay)
print("\nGEOTECNICA:")
print(df_geotecnica)

=== DATI CARICATI ===

SONDAGGI:
  ID_Sondaggio        Data  Sito Profondità_m   Metodo
0       DH-101  2024-01-15  Nord          150       RC
1       dh-101  20/02/2024  nord        200.5       rc
2       DH102   2024.03.10   SUD         zero  Diamond
3       DH-103  2024-04-05   sud          175  DIAMOND
4       DH-104  15/05/2024   Est          250       RC

ASSAY:
  ID_Campione ID_Sondaggio  Da_m   A_m Au_ppm Cu_ppm
0       S-101       DH-101   0.0   1.0    0.3    120
1       s-101        DH101  10.5  11.5    1.5    250
2        S102       DH-102    15    16     ND     ND
3       S103        DH-102    20    25    2.1     85
4        S104       DH-103    25    30    0.8    150
5        S105       DH-104    30    35    1.2    200

GEOTECNICA:
  ID_Test ID_Sondaggio Profondità_m RQD Tipo_Roccia
0   GT-01       DH-101           25  75     Granito
1   gt-01        DH101         50.5  85     granito
2    GT02       DH-102           75  ND      SCISTO
3   GT-03       DH-103         zero  

In [2]:
q_pulizia_sondaggi = """
SELECT
 
 -- fix ID_Sondaggio
CASE
WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'DH[0-9]*' THEN SUBSTR(ID_Sondaggio,1,2) || '-' || SUBSTR(ID_Sondaggio,3,3)
ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

-- fix Data
CASE 
WHEN Data LIKE '__/__/____' THEN SUBSTR(Data,7,4) || '-' || SUBSTR(Data,4,2) || '-' || SUBSTR(Data,1,2)
WHEN Data LIKE '%.%' THEN REPLACE(Data,'.','-')
ELSE TRIM(Data)
END AS data,

-- fix Sito
UPPER(TRIM(Sito)) AS sito,

-- fix Profondita'_m
CASE 
WHEN "Profondità_m" = 'zero' THEN 0.0
ELSE CAST(TRIM("Profondità_m") AS REAL) 
END AS profondita_m,

-- fix Metodo
 UPPER(TRIM(Metodo)) AS metodo

FROM sondaggi;
"""

cursor.execute("CREATE TEMPORARY TABLE sondaggi_puliti_temp AS" + q_pulizia_sondaggi)
print("=== SONDAGGI Table pulita ===")
df_verifica_sondaggi = pd.read_sql_query("SELECT * FROM sondaggi_puliti_temp", conn)
print(df_verifica_sondaggi)
print("\n")

q_pulizia_assay = """
SELECT
 
 -- fix ID_Campione
CASE
WHEN UPPER(TRIM(ID_Campione)) GLOB 'S[0-9]*' THEN SUBSTR(ID_Campione,1,1) || '-' || SUBSTR(ID_Campione,2,4)
ELSE UPPER(TRIM(ID_Campione))
END AS id_campione,

-- fix ID_Sondaggio
CASE
WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'DH[0-9]*' THEN SUBSTR(ID_Sondaggio,1,2) || '-' || SUBSTR(ID_Sondaggio,3,5)
ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

-- fix Da_m
CAST(TRIM(Da_m) AS REAL) AS da_m,

 -- fix A_m
CAST(TRIM(A_m) AS REAL) AS a_m,

-- fix Au_ppm
CASE 
WHEN Au_ppm = 'ND' THEN NULL
ELSE CAST(TRIM(Au_ppm) AS REAL) 
END AS Au_ppm,

-- fix Cu_ppm
CASE 
WHEN Au_ppm = 'ND' THEN NULL
ELSE CAST(TRIM(Cu_ppm) AS REAL) 
END AS Cu_ppm

FROM assay;
"""

cursor.execute("CREATE TEMPORARY TABLE assay_puliti_temp AS" + q_pulizia_assay)
print("=== ASSAY Table pulita ===")
df_verifica_assay = pd.read_sql_query("SELECT * FROM assay_puliti_temp", conn)
print(df_verifica_assay)
print("\n")

q_pulizia_geotecnica = """
SELECT
 
-- fix ID_Test
CASE
    WHEN UPPER(TRIM(ID_Test)) GLOB 'GT[0-9]*'
        THEN SUBSTR(UPPER(TRIM(ID_Test)), 1, 2) || '-' || SUBSTR(UPPER(TRIM(ID_Test)), 3, 4)
    ELSE UPPER(TRIM(ID_Test))
END AS id_test,

-- fix ID_Sondaggio
CASE
    WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'DH[0-9]*'
        THEN SUBSTR(UPPER(TRIM(ID_Sondaggio)), 1, 2) || '-' || SUBSTR(UPPER(TRIM(ID_Sondaggio)), 3, 5)
    ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

-- fix Profondità_m
CASE 
    WHEN LOWER(TRIM("Profondità_m")) = 'zero' THEN 0.0
    ELSE CAST(TRIM("Profondità_m") AS REAL)
END AS profondita_m,

-- fix RQD
CASE 
    WHEN UPPER(TRIM(RQD)) = 'ND' THEN NULL
    ELSE CAST(TRIM(RQD) AS INTEGER)
END AS rqd_clean,

-- fix Tipo_Roccia
UPPER(TRIM(Tipo_Roccia)) AS tipo_roccia

FROM geotecnica
"""

cursor.execute("CREATE TEMPORARY TABLE geotecnica_puliti_temp AS" + q_pulizia_geotecnica)
print("=== GEOTECNICA Table pulita ===")
df_verifica_geotecnica = pd.read_sql_query("SELECT * FROM geotecnica_puliti_temp", conn)
print(df_verifica_geotecnica)

=== SONDAGGI Table pulita ===
  id_sondaggio        data  sito  profondita_m   metodo
0       DH-101  2024-01-15  NORD         150.0       RC
1       DH-101  2024-02-20  NORD         200.5       RC
2       DH-102  2024-03-10   SUD           0.0  DIAMOND
3       DH-103  2024-04-05   SUD         175.0  DIAMOND
4       DH-104  2024-05-15   EST         250.0       RC


=== ASSAY Table pulita ===
  id_campione id_sondaggio  da_m   a_m  Au_ppm  Cu_ppm
0       S-101       DH-101   0.0   1.0     0.3   120.0
1       S-101       DH-101  10.5  11.5     1.5   250.0
2       S-102       DH-102  15.0  16.0     NaN     NaN
3      S-103        DH-102  20.0  25.0     2.1    85.0
4       S-104       DH-103  25.0  30.0     0.8   150.0
5       S-105       DH-104  30.0  35.0     1.2   200.0


=== GEOTECNICA Table pulita ===
  id_test id_sondaggio  profondita_m  rqd_clean tipo_roccia
0   GT-01       DH-101          25.0       75.0     GRANITO
1   GT-01       DH-101          50.5       85.0     GRANITO
2   GT

In [3]:
B1 = """
SELECT
    s.id_sondaggio,
    s.sito,
    s.profondita_m,
    s.metodo,
    COUNT(a.id_campione) AS num_campioni,
    ROUND(AVG(a.Au_ppm),2) AS media_Au_ppm,
    COUNT(g.id_test) AS num_test,
    ROUND(AVG(g.rqd_clean),2) AS media_RQD
FROM sondaggi_puliti_temp AS s
INNER JOIN assay_puliti_temp AS a ON s.id_sondaggio = a.id_sondaggio
INNER JOIN geotecnica_puliti_temp AS g ON s.id_sondaggio = g.id_sondaggio
GROUP BY s.id_sondaggio, s.sito, s.profondita_m, s.metodo;
"""

cursor.execute("CREATE TEMPORARY TABLE B1 AS" + B1)
print("=== Report completo sondaggi ===")
df_B1 = pd.read_sql_query("SELECT * FROM B1", conn)
print(df_B1)
print("\n")

B2 = """
WITH average_parameters AS (
    SELECT
        s.sito,
        ROUND(AVG(a.Au_ppm),2) AS media_Au_ppm,
        ROUND(AVG(g.rqd_clean),2) AS media_RQD
    FROM sondaggi_puliti_temp AS s
    INNER JOIN assay_puliti_temp AS a ON s.id_sondaggio = a.id_sondaggio
    INNER JOIN geotecnica_puliti_temp AS g ON s.id_sondaggio = g.id_sondaggio
    GROUP BY s.sito
)
SELECT 
    sito,
    media_RQD,
    CASE 
        WHEN media_RQD > 75 THEN 'Buona'
        WHEN media_RQD >= 50 AND media_RQD <= 75 THEN 'Media'
        WHEN media_RQD < 50 THEN 'Scarsa'
        ELSE 'Non classificabile'
    END AS qualita_roccia,  -- alias per la colonna
    media_Au_ppm
FROM average_parameters
ORDER BY media_RQD DESC;  -- opzionale: ordina per qualità
"""

cursor.execute("CREATE TEMPORARY TABLE B2 AS" + B2)
print("=== Classifica siti per qualità roccia ===")
df_B2 = pd.read_sql_query("SELECT * FROM B2", conn)
print(df_B2)
print("\n")

B3 = """
WITH assay_avg AS (
    SELECT 
        id_sondaggio,
        ROUND(AVG(Au_ppm), 2) AS media_Au_ppm
    FROM assay_puliti_temp
    GROUP BY id_sondaggio
),
geotech_avg AS (
    SELECT 
        id_sondaggio,
        ROUND(AVG(rqd_clean), 2) AS media_rqd
    FROM geotecnica_puliti_temp
    GROUP BY id_sondaggio
)
SELECT
    a.id_sondaggio,
    a.media_Au_ppm,
    g.media_rqd
FROM assay_avg a
INNER JOIN geotech_avg g
    ON a.id_sondaggio = g.id_sondaggio
WHERE g.media_rqd < 75 AND a.media_Au_ppm > 1.0
ORDER BY g.media_rqd DESC;
"""

cursor.execute("CREATE TEMPORARY TABLE B3 AS" + B3)
print("=== Anomalie da investigare ===")
df_B3 = pd.read_sql_query("SELECT * FROM B3", conn)
print(df_B3)
print("\n")

B4 = """
WITH au_max_per_sito AS (
    SELECT 
        s.sito,
        s.id_sondaggio,
        a.Au_ppm,
        ROUND(AVG((a.da_m + a.a_m) / 2.0), 2) AS profondita_media,
        ROW_NUMBER() OVER (PARTITION BY s.sito ORDER BY a.Au_ppm DESC) AS rn
    FROM sondaggi_puliti_temp s
    INNER JOIN assay_puliti_temp a ON s.id_sondaggio = a.id_sondaggio
    WHERE a.Au_ppm IS NOT NULL
    GROUP BY s.sito, s.id_sondaggio, a.da_m, a.a_m, a.Au_ppm
),
profondita_rqd AS (
    SELECT 
        am.sito,
        am.profondita_media AS profondita_media_Au_max,
        am.Au_ppm,
        g.rqd_clean,
        g.profondita_m,
        ROW_NUMBER() OVER (PARTITION BY am.sito ORDER BY ABS(g.profondita_m - am.profondita_media)) AS dist_rn
    FROM au_max_per_sito am
    LEFT JOIN geotecnica_puliti_temp g ON g.id_sondaggio = am.id_sondaggio
    WHERE am.rn = 1
)
SELECT 
    sito,
    profondita_media_Au_max,
    rqd_clean AS RQD_corrispondente
FROM profondita_rqd
WHERE dist_rn = 1 AND rqd_clean IS NOT NULL;
"""

cursor.execute("CREATE TEMPORARY TABLE B4 AS" + B4)
print("=== Profondità ottimale ===")
df_B4 = pd.read_sql_query("SELECT * FROM B4", conn)
print(df_B4)
print("\n")

B5 = """
SELECT
s.metodo AS tecnica,
COUNT(DISTINCT(s.id_sondaggio)) AS num_sondaggi,
ROUND(AVG(a.Au_ppm), 2) AS media_Au_ppm,
ROUND(AVG(g.rqd_clean), 2) AS media_RQD,
ROUND(AVG(g.profondita_m), 2) AS media_profondita
FROM sondaggi_puliti_temp AS s
INNER JOIN assay_puliti_temp AS a
ON s.id_sondaggio = a.id_sondaggio
INNER JOIN geotecnica_puliti_temp AS g
ON s.id_sondaggio = g.id_sondaggio
GROUP BY s.metodo;
"""

cursor.execute("CREATE TEMPORARY TABLE B5 AS" + B5)
print("=== Efficienza perforazione ===")
df_B5 = pd.read_sql_query("SELECT * FROM B5", conn)
print(df_B5)

=== Report completo sondaggi ===
  id_sondaggio  sito  profondita_m   metodo  num_campioni  media_Au_ppm  \
0       DH-101  NORD         150.0       RC             4           0.9   
1       DH-101  NORD         200.5       RC             4           0.9   
2       DH-102   SUD           0.0  DIAMOND             2           2.1   
3       DH-103   SUD         175.0  DIAMOND             1           0.8   
4       DH-104   EST         250.0       RC             1           1.2   

   num_test  media_RQD  
0         4       80.0  
1         4       80.0  
2         2        NaN  
3         1       92.0  
4         1       65.0  


=== Classifica siti per qualità roccia ===
   sito  media_RQD qualita_roccia  media_Au_ppm
0   SUD       92.0          Buona          1.45
1  NORD       80.0          Buona          0.90
2   EST       65.0          Media          1.20


=== Anomalie da investigare ===
  id_sondaggio  media_Au_ppm  media_rqd
0       DH-104           1.2       65.0


=== Profondit

## Considerazioni emerse dalle analisi statistiche:
    
### **A. Pulizia Dati**
- Dati globalmente puliti con minima presenza di valori anomali e/o indeterminati.

### **B. Analisi**

**B1. Report completo sondaggi**
- Profondità globale di indagine tra 150 e 250 metri
- Media Oro tra 0,9 e 2,1 ppm
- RQD tra 65 e 80

**B2. Classifica siti per qualità roccia**  
- Il settore EST mostra una roccia di qualità media con tenore di mineralizzazione medio-alto (1,20/1,45). 
- Il settore NORD mostra un elevato grado di fratturazione con un buon RQD.  
- Questo esclude quindi una mineralizzazione post-fratturazione, ma piuttosto contemporanea alla genesi litologica.

**B3. Anomalie da investigare**  
- Settore EST con roccia scadente ma mineralizzata. 
- Il tenore di mineralizzazione medio-alto indica una corsia preferenziale di scavo potenzialmente più rapido e meno problematico per via del grado di fratturazione in situ.

**B4. Profondità ottimale**  
- Settore EST con tenori di mineralizzazione massimi di 32,5 ppm è il punto primario di inizio scavo almeno fino a profondità? (dato mancante)

**B5. Efficienza perforazione**  
- L'attrezzatura a punta diamantata mostra un grado di efficienza leggermente superiore, a profondità simili ma RQD medio decisamente superiore (92 contro 78,3). 
- Rimane da valutare il reale beneficio in termini di costi/usura della punta.